In [12]:
from datetime import date
import time

import pandas as pd
import requests
import yfinance as yf


# ====== CONFIGURACIÓN ======
TICKERS = ["AAPL", "MSFT"]
# Añade o quita tickers
EMAIL = "tu-correo@example.com"       # Sustituye por tu correo
ARCHIVO_SALIDA = "fundamentales_tickers.csv"

HEADERS = {
    "User-Agent": f"Quantfolio {EMAIL}",
    "Accept-Encoding": "gzip, deflate",
}


def obtener_mapa_tickers():
    """Descarga la lista de empresas y CIK publicada por la SEC."""
    url = "https://www.sec.gov/files/company_tickers.json"
    respuesta = requests.get(url, headers=HEADERS, timeout=30)
    respuesta.raise_for_status()

    mapa = {}
    for empresa in respuesta.json().values():
        ticker = empresa.get("ticker", "").upper()
        mapa[ticker] = str(empresa["cik_str"]).zfill(10)

    return mapa


def obtener_ultimo_dato_anual(facts, tags):
    """Busca el dato anual más reciente expresado en USD."""
    formularios_anuales = {"10-K", "20-F", "40-F"}

    for taxonomia, tag in tags:
        concepto = facts.get(taxonomia, {}).get(tag)
        if not concepto:
            continue

        observaciones = concepto.get("units", {}).get("USD", [])
        anuales = []

        for obs in observaciones:
            if obs.get("form") not in formularios_anuales:
                continue
            if "start" not in obs or "end" not in obs:
                continue

            inicio = date.fromisoformat(obs["start"])
            fin = date.fromisoformat(obs["end"])
            duracion = (fin - inicio).days

            # Filtra periodos de aproximadamente un año.
            if 300 <= duracion <= 400:
                anuales.append(obs)

        if anuales:
            ultimo = max(
                anuales,
                key=lambda obs: (obs.get("end", ""), obs.get("filed", "")),
            )
            return {
                "valor": ultimo["val"],
                "fecha": ultimo["end"],
            }

    return None


def obtener_acciones_en_circulacion(facts):
    """Busca las acciones en circulación más recientes reportadas a la SEC."""
    observaciones = (
        facts.get("dei", {})
        .get("EntityCommonStockSharesOutstanding", {})
        .get("units", {})
        .get("shares", [])
    )

    formularios_validos = {"10-K", "10-Q", "20-F", "40-F"}
    observaciones = [
        obs for obs in observaciones
        if obs.get("form") in formularios_validos and obs.get("end")
    ]

    if not observaciones:
        return None

    return max(
        observaciones,
        key=lambda obs: (obs.get("end", ""), obs.get("filed", "")),
    )


def analizar_ticker(ticker, mapa_tickers):
    ticker = ticker.upper()

    if ticker not in mapa_tickers:
        raise ValueError("Ticker no encontrado en el directorio de la SEC.")

    cik = mapa_tickers[ticker]

    # Datos XBRL de la empresa publicados por la SEC.
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    respuesta = requests.get(url, headers=HEADERS, timeout=30)
    respuesta.raise_for_status()
    facts = respuesta.json()["facts"]

    ingresos = obtener_ultimo_dato_anual(
        facts,
        [
            ("us-gaap", "RevenueFromContractWithCustomerExcludingAssessedTax"),
            ("us-gaap", "Revenues"),
            ("us-gaap", "SalesRevenueNet"),
            ("ifrs-full", "Revenue"),
        ],
    )

    beneficio = obtener_ultimo_dato_anual(
        facts,
        [
            ("us-gaap", "NetIncomeLoss"),
            ("ifrs-full", "ProfitLoss"),
        ],
    )

    acciones_info = obtener_acciones_en_circulacion(facts)

    if ingresos is None:
        raise ValueError("No se encontraron ingresos anuales.")
    if beneficio is None:
        raise ValueError("No se encontró el beneficio neto anual.")
    if acciones_info is None:
        raise ValueError("No se encontraron acciones en circulación.")

    # Último cierre disponible.
    historial = yf.Ticker(ticker).history(
        period="5d",
        auto_adjust=False,
    )
    cierres = historial["Close"].dropna()

    if cierres.empty:
        raise ValueError("No se pudo obtener el precio desde Yahoo Finance.")

    precio = float(cierres.iloc[-1])
    fecha_precio = cierres.index[-1].date()
    acciones = acciones_info["val"]

    capitalizacion = precio * acciones
    ventas = ingresos["valor"]
    beneficio_neto = beneficio["valor"]

    price_to_sales = capitalizacion / ventas if ventas > 0 else None
    price_to_earnings = (
        capitalizacion / beneficio_neto if beneficio_neto > 0 else None
    )

    return {
        "ticker": ticker,
        "cik": cik,
        "precio_usd": precio,
        "fecha_precio": str(fecha_precio),
        "acciones_reportadas": acciones,
        "fecha_dato_acciones": acciones_info["end"],
        "capitalizacion_aprox_usd": capitalizacion,
        "ingresos_usd": ventas,
        "fecha_ingresos": ingresos["fecha"],
        "beneficio_neto_usd": beneficio_neto,
        "fecha_beneficio": beneficio["fecha"],
        "price_to_sales": price_to_sales,
        "price_to_earnings": price_to_earnings,
        "error": "",
    }


def main():
    mapa_tickers = obtener_mapa_tickers()
    resultados = []

    for ticker in TICKERS:
        try:
            fila = analizar_ticker(ticker, mapa_tickers)
            print(f"OK: {ticker}")
        except Exception as error:
            fila = {
                "ticker": ticker.upper(),
                "cik": None,
                "precio_usd": None,
                "fecha_precio": None,
                "acciones_reportadas": None,
                "fecha_dato_acciones": None,
                "capitalizacion_aprox_usd": None,
                "ingresos_usd": None,
                "fecha_ingresos": None,
                "beneficio_neto_usd": None,
                "fecha_beneficio": None,
                "price_to_sales": None,
                "price_to_earnings": None,
                "error": str(error),
            }
            print(f"ERROR en {ticker}: {error}")

        resultados.append(fila)

        # Pausa breve entre empresas para espaciar las consultas.
        time.sleep(0.2)

    tabla = pd.DataFrame(resultados)
    print("\nResultados:")
    print(tabla.to_string(index=False))

    tabla.to_csv(ARCHIVO_SALIDA, index=False, encoding="utf-8-sig")
    print(f"\nArchivo guardado en: {ARCHIVO_SALIDA}")
    print(
        "Nota: capitalización aproximada = último cierre × acciones del último "
        "informe. P/S y P/E usan el último ejercicio anual, no TTM."
    )


if __name__ == "__main__":
    main()

OK: AAPL
OK: MSFT

Resultados:
ticker        cik  precio_usd fecha_precio  acciones_reportadas fecha_dato_acciones  capitalizacion_aprox_usd  ingresos_usd fecha_ingresos  beneficio_neto_usd fecha_beneficio  price_to_sales  price_to_earnings error
  AAPL 0000320193  342.369995   2026-11-06          14532182000          2026-10-23              4.975383e+12  478834000000     2026-09-26        132175000000      2026-09-26       10.390622          37.642391      
  MSFT 0000789019  530.609985   2026-11-06           7426194552          2026-10-22              3.940413e+12  331839000000     2026-06-30        133749000000      2026-06-30       11.874472          29.461252      

Archivo guardado en: fundamentales_tickers.csv
Nota: capitalización aproximada = último cierre × acciones del último informe. P/S y P/E usan el último ejercicio anual, no TTM.
